# SemLightGCL — Complete Kaggle Implementation

## Cell 1 — Environment Setup

In [1]:
# ─── Install dependencies ────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentence-transformers', '-q'])

# ─── Clone LightGCL official repo ───────────────────────────────────────────
import os
WORK = '/kaggle/working'
REPO = f'{WORK}/LightGCL'

if not os.path.exists(REPO):
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/HKUDS/LightGCL.git', REPO],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print('Repo already cloned.')

sys.path.insert(0, REPO)
os.chdir(REPO)

# ─── GPU check ───────────────────────────────────────────────────────────────
import torch
print(f'PyTorch  : {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected — enable GPU in Kaggle Settings!')

# ─── Show repo structure ─────────────────────────────────────────────────────
py_files = sorted([f for f in os.listdir('.') if f.endswith('.py')])
print(f'\nRepo .py files: {py_files}')
print(f'Data dirs     : {os.listdir("data") if os.path.exists("data") else "data/ not found"}')

Cloning into '/kaggle/working/LightGCL'...

PyTorch  : 2.10.0+cu128
GPU      : Tesla T4
VRAM     : 15.6 GB

Repo .py files: ['main.py', 'model.py', 'parser.py', 'utils.py']
Data dirs     : ['amazon.zip', 'gowalla', 'ml10m.zip', 'tmall.zip', 'yelp']


In [2]:
import os, sys, subprocess, zipfile, pickle, requests
import numpy as np
import torch
import torch.nn.functional as F
import scipy.sparse as sp

REPO = '/kaggle/working/LightGCL'
DATA = f'{REPO}/data'

# ── 1. Re-clone if wiped ──────────────────────────────────────────────────────
if not os.path.exists(REPO):
    print("Re-cloning LightGCL...")
    subprocess.run(['git','clone','https://github.com/HKUDS/LightGCL.git', REPO], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)

# ── 2. Extract existing zips (yelp, gowalla, amazon) ─────────────────────────
for fname in os.listdir(DATA):
    if fname.endswith('.zip'):
        out = os.path.join(DATA, fname.replace('.zip',''))
        if not os.path.exists(out):
            print(f"Extracting {fname}...")
            with zipfile.ZipFile(os.path.join(DATA, fname)) as z:
                z.extractall(DATA)

# ── 3. Download & preprocess ML-10M ──────────────────────────────────────────
def preprocess_ml10m():
    out_dir = os.path.join(DATA, 'ml-10m')
    if os.path.exists(os.path.join(out_dir, 'trnMat.pkl')):
        print('[ml-10m] Already done — skip'); return
    os.makedirs(out_dir, exist_ok=True)
    zip_path = os.path.join(out_dir, 'ml-10m.zip')
    if not os.path.exists(zip_path):
        print('[ml-10m] Downloading...')
        r = requests.get('https://files.grouplens.org/datasets/movielens/ml-10m.zip', stream=True)
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
    with zipfile.ZipFile(zip_path) as z:
        z.extract('ml-10M100K/ratings.dat', out_dir)
    print('[ml-10m] Preprocessing...')
    from collections import defaultdict
    import random; random.seed(42)
    user_items = defaultdict(set)
    with open(os.path.join(out_dir, 'ml-10M100K', 'ratings.dat')) as f:
        for line in f:
            u, i, r, _ = line.strip().split('::')
            if float(r) >= 4.0:
                user_items[int(u)].add(int(i))
    for _ in range(10):
        item_count = defaultdict(int)
        for items in user_items.values():
            for i in items: item_count[i] += 1
        user_items = {u: {i for i in items if item_count[i] >= 10}
                      for u, items in user_items.items()}
        user_items = {u: v for u, v in user_items.items() if len(v) >= 10}
    users = sorted(user_items.keys())
    items = sorted({i for s in user_items.values() for i in s})
    u2id  = {u: idx for idx, u in enumerate(users)}
    i2id  = {i: idx for idx, i in enumerate(items)}
    n_u, n_i = len(users), len(items)
    trn_rows, trn_cols, tst_rows, tst_cols = [], [], [], []
    for u, its in user_items.items():
        its = list(its); random.shuffle(its)
        cut = max(1, int(len(its) * 0.8))
        uid = u2id[u]
        for i in its[:cut]: trn_rows.append(uid); trn_cols.append(i2id[i])
        for i in its[cut:]: tst_rows.append(uid); tst_cols.append(i2id[i])
    trn = sp.csr_matrix((np.ones(len(trn_rows)), (trn_rows, trn_cols)), shape=(n_u, n_i))
    tst = sp.csr_matrix((np.ones(len(tst_rows)), (tst_rows, tst_cols)), shape=(n_u, n_i))
    with open(os.path.join(out_dir, 'trnMat.pkl'), 'wb') as f: pickle.dump(trn, f)
    with open(os.path.join(out_dir, 'tstMat.pkl'), 'wb') as f: pickle.dump(tst, f)
    print(f'[ml-10m] Done  users={n_u:,}  items={n_i:,}  trn_nnz={trn.nnz:,}  tst_nnz={tst.nnz:,}')

# ── 4. Download Tmall ─────────────────────────────────────────────────────────
def preprocess_tmall():
    out_dir = os.path.join(DATA, 'tmall')
    if os.path.exists(os.path.join(out_dir, 'trnMat.pkl')):
        print('[tmall] Already done — skip'); return
    os.makedirs(out_dir, exist_ok=True)
    print('[tmall] Downloading...')
    base = 'https://raw.githubusercontent.com/wujcan/SGL-Torch/main/data/tmall'
    for fname in ['trnMat.pkl', 'tstMat.pkl']:
        r = requests.get(f'{base}/{fname}')
        if r.status_code == 200:
            with open(os.path.join(out_dir, fname), 'wb') as f: f.write(r.content)
            print(f'[tmall] Downloaded {fname}')
        else:
            print(f'[tmall] Failed {fname} — status {r.status_code}'); return
    with open(os.path.join(out_dir, 'trnMat.pkl'), 'rb') as f:
        trn = pickle.load(f).tocsr()
    print(f'[tmall] Done  users={trn.shape[0]:,}  items={trn.shape[1]:,}  nnz={trn.nnz:,}')

preprocess_ml10m()
preprocess_tmall()

# ── 5. Write precompute_sem.py ────────────────────────────────────────────────
precompute_code = '''
import argparse, os, pickle
import numpy as np
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

def load_adj(data_dir):
    with open(os.path.join(data_dir, "trnMat.pkl"), "rb") as f:
        import scipy.sparse as sp
        adj = pickle.load(f).tocsr()
    n_users, n_items = adj.shape
    print(f"  trnMat  users={n_users:,}  items={n_items:,}  nnz={adj.nnz:,}")
    return adj, n_users, n_items

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset",   type=str, default="yelp")
    parser.add_argument("--data_path", type=str, default="./data")
    parser.add_argument("--model",     type=str, default="intfloat/e5-small-v2")
    parser.add_argument("--batch",     type=int, default=256)
    args = parser.parse_args()
    data_dir = os.path.join(args.data_path, args.dataset)
    print(f"Dataset: {args.dataset}")
    adj, n_users, n_items = load_adj(data_dir)
    item_strings = [f"passage: item {i}" for i in range(n_items)]
    print(f"Loading {args.model}...")
    encoder = SentenceTransformer(args.model)
    print(f"Encoding {n_items:,} items...")
    item_np  = encoder.encode(item_strings, batch_size=args.batch,
                               show_progress_bar=True,
                               normalize_embeddings=True, device="cpu")
    item_emb = torch.FloatTensor(item_np)
    import numpy as np
    deg      = torch.FloatTensor(np.asarray(adj.sum(1)).flatten()).clamp(min=1).unsqueeze(1)
    user_np  = (adj @ item_emb.numpy()) / deg.numpy()
    user_emb = F.normalize(torch.FloatTensor(user_np), dim=-1)
    torch.save(item_emb, os.path.join(data_dir, "item_sem_embeds.pt"))
    torch.save(user_emb, os.path.join(data_dir, "user_sem_embeds.pt"))
    print(f"item_sem_embeds.pt  {item_emb.shape}")
    print(f"user_sem_embeds.pt  {user_emb.shape}")

if __name__ == "__main__":
    main()
'''.strip()
with open(f'{REPO}/precompute_sem.py', 'w') as f:
    f.write(precompute_code)
print("precompute_sem.py written")

# ── 6. Write main_sem.py ──────────────────────────────────────────────────────
main_sem_code = '''
import argparse, os, sys, time, pickle, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import scipy.sparse as sp

REPO = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, REPO)

def get_args():
    p = argparse.ArgumentParser("SemLightGCL")
    p.add_argument("--dataset",    default="yelp")
    p.add_argument("--data_path",  default="./data")
    p.add_argument("--emb_dim",    type=int,   default=64)
    p.add_argument("--layers",     type=int,   default=2)
    p.add_argument("--q",          type=int,   default=5)
    p.add_argument("--lambda1",    type=float, default=0.2)
    p.add_argument("--lambda2",    type=float, default=0.1)
    p.add_argument("--tau1",       type=float, default=0.2)
    p.add_argument("--tau2",       type=float, default=0.2)
    p.add_argument("--gamma",      type=float, default=0.5)
    p.add_argument("--lr",         type=float, default=1e-3)
    p.add_argument("--decay",      type=float, default=1e-4)
    p.add_argument("--batch_size", type=int,   default=2048)
    p.add_argument("--epochs",     type=int,   default=1000)
    p.add_argument("--patience",   type=int,   default=30)
    p.add_argument("--topk",       type=int,   default=20)
    p.add_argument("--ablation",   default="full",
                   choices=["full","no_sem","no_ada","baseline"])
    p.add_argument("--device",     default="cuda")
    return p.parse_args()

def load_data(data_dir):
    with open(os.path.join(data_dir, "trnMat.pkl"), "rb") as f:
        trn = pickle.load(f).tocsr()
    with open(os.path.join(data_dir, "tstMat.pkl"), "rb") as f:
        tst = pickle.load(f).tocsr()
    n_users, n_items = trn.shape
    train_dict, test_dict = {}, {}
    for u in range(n_users):
        ti = trn.getrow(u).indices.tolist()
        if ti: train_dict[u] = ti
        te = tst.getrow(u).indices.tolist()
        if te: test_dict[u]  = te
    print(f"  users={n_users:,}  items={n_items:,}  train_nnz={trn.nnz:,}  test_nnz={tst.nnz:,}")
    return trn, tst, train_dict, test_dict, n_users, n_items

def build_norm_adj(trn_csr, n_users, n_items, device):
    n = n_users + n_items
    zeros_uu = sp.csr_matrix((n_users, n_users))
    zeros_ii = sp.csr_matrix((n_items, n_items))
    A = sp.bmat([[zeros_uu, trn_csr], [trn_csr.T, zeros_ii]], format="csr")
    d   = np.asarray(A.sum(1)).flatten() + 1e-8
    D05 = sp.diags(1.0 / np.sqrt(d))
    A_norm = (D05 @ A @ D05).tocoo()
    idx = torch.LongTensor(np.stack([A_norm.row, A_norm.col]))
    val = torch.FloatTensor(A_norm.data)
    return torch.sparse_coo_tensor(idx, val, (n, n)).to(device)

def svd_augment(trn_csr, rank, device):
    import scipy.sparse.linalg as la
    print(f"  Computing SVD at rank={rank}...")
    U, S, Vt = la.svds(trn_csr.astype(np.float32), k=rank)
    return (torch.FloatTensor(U.copy()).to(device),
            torch.FloatTensor(S.copy()).to(device),
            torch.FloatTensor(Vt.copy()).to(device))

class LightGCL(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers,
                 adj_norm, U_svd, S_svd, Vt_svd):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        self.adj_norm = adj_norm
        self.register_buffer("U_svd",  U_svd)
        self.register_buffer("S_svd",  S_svd)
        self.register_buffer("Vt_svd", Vt_svd)

    def forward(self):
        x    = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        outs = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(self.adj_norm, x)
            outs.append(x)
        x   = torch.stack(outs, dim=1).mean(dim=1)
        h_u = x[:self.n_users]
        h_i = x[self.n_users:]
        tmp       = self.Vt_svd @ self.item_emb.weight
        h_tilde_u = F.normalize(self.U_svd @ (self.S_svd.unsqueeze(1) * tmp) + self.user_emb.weight, dim=-1)
        tmp2      = self.U_svd.T @ self.user_emb.weight
        h_tilde_i = F.normalize(self.Vt_svd.T @ (self.S_svd.unsqueeze(1) * tmp2) + self.item_emb.weight, dim=-1)
        return h_u, h_i, h_tilde_u, h_tilde_i

def bpr(h_u, h_i, uids, pos, neg, decay):
    u  = h_u[uids]; pi = h_i[pos]; ni = h_i[neg]
    loss = -F.logsigmoid((u*pi).sum(-1) - (u*ni).sum(-1)).mean()
    reg  = (u.norm(2).pow(2) + pi.norm(2).pow(2) + ni.norm(2).pow(2)) / (2 * len(uids))
    return loss + decay * reg

def info_nce(a, b, temp):
    a = F.normalize(a, dim=-1); b = F.normalize(b, dim=-1)
    N = a.size(0)
    s = a @ b.T / temp
    l = torch.arange(N, device=a.device)
    return (F.cross_entropy(s, l) + F.cross_entropy(s.T, l)) / 2

def adaptive_weights(trn_csr, gamma, device):
    u_deg = torch.FloatTensor(np.asarray(trn_csr.sum(1)).flatten())
    i_deg = torch.FloatTensor(np.asarray(trn_csr.sum(0)).flatten())
    def wfn(d):
        w = 1.0 / torch.pow(torch.log(d + math.e), gamma)
        return (w / w.mean()).to(device)
    return wfn(u_deg), wfn(i_deg)

@torch.no_grad()
def evaluate(model, train_dict, test_dict, n_items, device, topk=20):
    model.eval()
    h_u, h_i, _, _ = model.forward()
    test_users = [u for u in test_dict if test_dict[u]]
    recalls, ndcgs = [], []
    for start in range(0, len(test_users), 512):
        batch  = test_users[start:start+512]
        uids   = torch.LongTensor(batch).to(device)
        scores = (h_u[uids] @ h_i.T).cpu().numpy()
        for i, uid in enumerate(batch):
            for iid in train_dict.get(uid, []):
                scores[i, iid] = -1e9
            top  = np.argpartition(scores[i], -topk)[-topk:]
            top  = top[np.argsort(-scores[i, top])]
            gt   = set(test_dict[uid])
            hits = [1 if p in gt else 0 for p in top]
            recalls.append(sum(hits) / max(len(gt), 1))
            dcg   = sum(h / math.log2(r+2) for r, h in enumerate(hits))
            ideal = sum(1/math.log2(r+2) for r in range(min(len(gt), topk)))
            ndcgs.append(dcg / max(ideal, 1e-8))
    model.train()
    return float(np.mean(recalls)), float(np.mean(ndcgs))

def train():
    args   = get_args()
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    print(f"{'='*60}")
    print(f"SemLightGCL | {args.dataset} | ablation={args.ablation} | {device}")
    print(f"{'='*60}")
    data_dir = os.path.join(args.data_path, args.dataset)
    trn_csr, tst_csr, train_dict, test_dict, n_users, n_items = load_data(data_dir)
    adj_norm              = build_norm_adj(trn_csr, n_users, n_items, device)
    U_svd, S_svd, Vt_svd = svd_augment(trn_csr, args.q, device)
    u_w, i_w              = adaptive_weights(trn_csr, args.gamma, device)
    model = LightGCL(n_users, n_items, args.emb_dim, args.layers,
                     adj_norm, U_svd, S_svd, Vt_svd).to(device)
    sem_ext = None
    if args.ablation != "baseline":
        item_path = os.path.join(data_dir, "item_sem_embeds.pt")
        if os.path.exists(item_path):
            from semlightgcl import SemExtension
            item_sem = torch.load(item_path, map_location="cpu")
            sem_ext  = SemExtension(
                item_sem_embeds=item_sem, adj=trn_csr,
                emb_dim=args.emb_dim, lambda2=args.lambda2, tau2=args.tau2,
                gamma=args.gamma if args.ablation != "no_ada" else 0.0,
                device=str(device),
            ).to(device)
            proj_n = sum(p.numel() for p in sem_ext.proj.parameters())
            print(f"  SemExtension ready  proj_params={proj_n:,}")
        else:
            print("  item_sem_embeds.pt not found — falling back to baseline")
            args.ablation = "baseline"
    params    = list(model.parameters())
    if sem_ext: params += list(sem_ext.proj.parameters())
    optimizer = optim.Adam(params, lr=args.lr)
    all_u = np.array([u  for u, items in train_dict.items() for _ in items])
    all_i = np.array([it for u, items in train_dict.items() for it in items])
    print(f"  Training pairs: {len(all_u):,}")
    best_r = best_n = 0.0
    patience = 0
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0      = time.time()
        perm    = np.random.permutation(len(all_u))
        ep_loss = 0.0
        n_batch = 0
        for s in range(0, len(all_u), args.batch_size):
            idx  = perm[s:s + args.batch_size]
            uids = torch.LongTensor(all_u[idx]).to(device)
            pos  = torch.LongTensor(all_i[idx]).to(device)
            neg  = torch.LongTensor(np.random.randint(0, n_items, len(idx))).to(device)
            h_u, h_i, h_tu, h_ti = model.forward()
            loss = bpr(h_u, h_i, uids, pos, neg, args.decay)
            uu = uids.unique()
            ui = pos.unique()
            if args.ablation == "baseline":
                cl   = (info_nce(h_u[uu], h_tu[uu], args.tau1) + info_nce(h_i[ui], h_ti[ui], args.tau1)) / 2
                loss = loss + args.lambda1 * cl
            else:
                ada_svd, sem = sem_ext(uu, ui, h_u[uu], h_i[ui], h_tu[uu], h_ti[ui], tau1=args.tau1)
                if args.ablation == "no_sem":
                    sem = torch.tensor(0., device=device)
                loss = loss + args.lambda1 * ada_svd + args.lambda2 * sem
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
            n_batch += 1
        r, n    = evaluate(model, train_dict, test_dict, n_items, device, args.topk)
        elapsed = int(time.time() - t0)
        print(f"Ep {epoch:3d}  loss={ep_loss/n_batch:.4f}  R@{args.topk}={r:.4f}  N@{args.topk}={n:.4f}  ({elapsed}s)")
        if r > best_r:
            best_r, best_n = r, n
            patience = 0
            torch.save(model.state_dict(), f"{args.dataset}_{args.ablation}_best.pt")
            print(f"  New best  R@{args.topk}={best_r:.4f}  N@{args.topk}={best_n:.4f}")
        else:
            patience += 1
            if patience >= args.patience:
                print(f"  Early stop at epoch {epoch}")
                break
    print(f"FINAL {args.dataset} [{args.ablation}]  R@{args.topk}={best_r:.4f}  N@{args.topk}={best_n:.4f}")
    return best_r, best_n

if __name__ == "__main__":
    train()
'''.strip()
with open(f'{REPO}/main_sem.py', 'w') as f:
    f.write(main_sem_code)
print("main_sem.py written")

# ── 7. Run precompute cho cả 5 dataset ───────────────────────────────────────
for ds in ['yelp', 'gowalla', 'amazon', 'ml-10m', 'tmall']:
    path = os.path.join(DATA, ds)
    if not os.path.isdir(path): print(f'[{ds}] missing — skip'); continue
    emb = os.path.join(path, 'item_sem_embeds.pt')
    if os.path.exists(emb):
        t = torch.load(emb, map_location='cpu')
        print(f'[{ds}] Embeddings exist {t.shape} — skip'); continue
    print(f'\n--- Precomputing: {ds} ---')
    subprocess.run([sys.executable, 'precompute_sem.py',
                    '--dataset', ds, '--data_path', './data', '--batch', '256'])

# ── 8. Verify tất cả ─────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"{'Dataset':<12} {'Users':>8} {'Items':>8} {'Train nnz':>11} {'Emb':>6}")
print(f"{'='*65}")
for ds in ['yelp', 'gowalla', 'amazon', 'ml-10m', 'tmall']:
    p = os.path.join(DATA, ds)
    if not os.path.isdir(p): print(f'{ds:<12} MISSING'); continue
    with open(os.path.join(p, 'trnMat.pkl'), 'rb') as f: trn = pickle.load(f).tocsr()
    emb_ok = 'Embedded successfully' if os.path.exists(os.path.join(p, 'item_sem_embeds.pt')) else 'Embedded failed'
    print(f'{ds:<12} {trn.shape[0]:>8,} {trn.shape[1]:>8,} {trn.nnz:>11,} {emb_ok:>6}')
print(f"{'='*65}")

Extracting amazon.zip...
Extracting ml10m.zip...
Extracting tmall.zip...
[ml-10m] Downloading...
[ml-10m] Preprocessing...
[ml-10m] Done  users=66,015  items=7,794  trn_nnz=3,949,773  tst_nnz=1,020,488
[tmall] Already done — skip
precompute_sem.py written
main_sem.py written

--- Precomputing: yelp ---


/kaggle/working/LightGCL/precompute_sem.py:10: DeprecationWarning: Please import `coo_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.coo` namespace is deprecated and will be removed in SciPy 2.0.0.
  adj = pickle.load(f).tocsr()
/kaggle/working/LightGCL/precompute_sem.py:10: DeprecationWarning: numpy.core._multiarray_umath is deprecated and has been renamed to numpy._core._multiarray_umath. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core._multiarray_umath._reconstruct.
  adj = pickle.load(f).tocsr()


Dataset: yelp
  trnMat  users=29,601  items=24,734  nnz=1,069,128
Loading intfloat/e5-small-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1453.93it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 24,734 items...


...

Encoding 41,390 items...


Batches: 100%|██████████| 162/162 [01:34<00:00,  1.71it/s]


item_sem_embeds.pt  torch.Size([41390, 384])
user_sem_embeds.pt  torch.Size([47939, 384])

Dataset         Users    Items   Train nnz    Emb
yelp           29,601   24,734   1,069,128 Embedded successfully
gowalla        50,821   57,440   1,172,425 Embedded successfully


/tmp/ipykernel_22/101298046.py:401: DeprecationWarning: Please import `coo_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.coo` namespace is deprecated and will be removed in SciPy 2.0.0.
  with open(os.path.join(p, 'trnMat.pkl'), 'rb') as f: trn = pickle.load(f).tocsr()
/tmp/ipykernel_22/101298046.py:401: DeprecationWarning: numpy.core._multiarray_umath is deprecated and has been renamed to numpy._core._multiarray_umath. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core._multiarray_umath._reconstruct.
  with open(os.path.join(p, 'trnMat.pkl'), 'rb') as f: trn = pickle.load(f).tocsr()


amazon         78,578   77,801   2,240,156 Embedded successfully
ml-10m         66,015    7,794   3,949,773 Embedded successfully
tmall          47,939   41,390   2,357,450 Embedded successfully


## Cell 2 — Write `semlightgcl.py` (Our Additions)

In [3]:
sem_code = '''
"""
semlightgcl.py  —  SemLightGCL additions to the LightGCL base model.

Adds:
  1. SemanticProjection : projects frozen E5-small-v2 embeddings into CF space
  2. adaptive_info_nce  : degree-weighted InfoNCE for the SVD contrastive loss
  3. SemExtension       : a module that holds both additions and computes
                          the combined extra loss  (L_cl_sem + adaptive L_cl_svd)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math


# ─── 1. Semantic projection layer ─────────────────────────────────────────────
class SemanticProjection(nn.Module):
    """Single linear layer: R^sem_dim → R^emb_dim (L2-normalized output)."""
    def __init__(self, sem_dim: int = 384, emb_dim: int = 64):
        super().__init__()
        self.fc = nn.Linear(sem_dim, emb_dim, bias=True)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.fc(x), dim=-1)


# ─── 2. Loss helpers ──────────────────────────────────────────────────────────
def info_nce(z1: torch.Tensor, z2: torch.Tensor, temp: float = 0.2) -> torch.Tensor:
    """Symmetric InfoNCE with in-batch negatives."""
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    N  = z1.size(0)
    sim = z1 @ z2.T / temp
    lbl = torch.arange(N, device=z1.device)
    return (F.cross_entropy(sim, lbl) + F.cross_entropy(sim.T, lbl)) / 2.0


def adaptive_info_nce(
    z1: torch.Tensor, z2: torch.Tensor,
    weights: torch.Tensor, temp: float = 0.2
) -> torch.Tensor:
    """
    Degree-adaptive InfoNCE: each sample i contributes weights[i] to the mean.
    Sparse nodes (low degree) get higher weight → more structural guidance.
    """
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    N  = z1.size(0)
    sim = z1 @ z2.T / temp
    lbl = torch.arange(N, device=z1.device)
    per = (
        F.cross_entropy(sim,   lbl, reduction="none") +
        F.cross_entropy(sim.T, lbl, reduction="none")
    ) / 2.0
    return (weights * per).mean()


# ─── 3. Weight computation (call once before training) ────────────────────────
def compute_adaptive_weights(
    adj,                          # scipy sparse or dense interaction matrix
    gamma: float = 0.5,
    device: str  = "cuda"
):
    """
    Returns (user_weights, item_weights): normalized tensors shaped
    (n_users,) and (n_items,) where mean=1 and sparse nodes > 1.
    Formula:  w(v) = 1 / log(deg(v) + e)^gamma  then normalized.
    """
    if hasattr(adj, "toarray"):
        u_deg = torch.FloatTensor(np.asarray(adj.sum(1)).flatten())
        i_deg = torch.FloatTensor(np.asarray(adj.sum(0)).flatten())
    else:
        u_deg = adj.float().sum(1)
        i_deg = adj.float().sum(0)

    def _w(deg):
        w = 1.0 / torch.pow(torch.log(deg + math.e), gamma)
        return (w / w.mean()).to(device)

    return _w(u_deg), _w(i_deg)


# ─── 4. Main extension module ─────────────────────────────────────────────────
class SemExtension(nn.Module):
    """
    Drop-in extension for ANY LightGCL implementation.

    Usage inside the training loop:
        extra_loss = sem_ext(user_ids, item_ids,
                             h_u, h_i,           # main graph embeddings
                             h_tilde_u, h_tilde_i,  # SVD embeddings
                             tau1=0.2)            # original CL temperature
        total_loss = bpr_loss + lambda1*cl_loss_original + lambda2*extra_loss

    BUT if you also want adaptive weighting on the SVD loss, skip the original
    cl_loss and call this instead:
        sem_loss, ada_svd_loss = sem_ext.split_losses(...)
        total_loss = bpr_loss + lambda1*ada_svd_loss + lambda2*sem_loss
    """

    def __init__(
        self,
        item_sem_embeds: torch.Tensor,   # (n_items, sem_dim)  — pre-computed
        adj,                             # scipy sparse interaction matrix
        emb_dim:  int   = 64,
        lambda2:  float = 0.1,
        tau2:     float = 0.2,
        gamma:    float = 0.5,
        device:   str   = "cuda",
    ):
        super().__init__()
        self.device  = device
        self.lambda2 = lambda2
        self.tau2    = tau2

        sem_dim = item_sem_embeds.size(1)
        self.proj = SemanticProjection(sem_dim, emb_dim).to(device)

        # Store item semantic embeddings
        self.register_buffer("item_sem", item_sem_embeds.to(device))

        # Derive user semantic embeddings by mean-pooling
        self.register_buffer("user_sem", self._mean_pool(adj, item_sem_embeds, device))

        # Pre-compute adaptive weights
        u_w, i_w = compute_adaptive_weights(adj, gamma, device)
        self.register_buffer("user_w", u_w)
        self.register_buffer("item_w", i_w)

    @staticmethod
    def _mean_pool(adj, item_sem, device):
        """s_u = mean of interacted items\' embeddings."""
        if hasattr(adj, "toarray"):
            A = torch.FloatTensor(np.asarray(adj.toarray()))
        else:
            A = adj.float()
        deg = A.sum(1, keepdim=True).clamp(min=1.0)
        user_sem = (A @ item_sem.cpu()) / deg
        return user_sem.to(device)

    def forward(
        self,
        user_ids:    torch.Tensor,
        item_ids:    torch.Tensor,
        h_u:         torch.Tensor,   # GNN embeddings (main graph)
        h_i:         torch.Tensor,
        h_tilde_u:   torch.Tensor,   # GNN embeddings (SVD graph)
        h_tilde_i:   torch.Tensor,
        tau1:        float = 0.2,
    ):
        """
        Returns:
            ada_svd_loss  — adaptive structural contrastive loss  (replaces original cl_loss)
            sem_loss      — semantic contrastive loss             (new term)
        """
        # ── Adaptive SVD contrastive loss ────────────────────────────────────
        w_u = self.user_w[user_ids]
        w_i = self.item_w[item_ids]
        ada_svd_loss = (
            adaptive_info_nce(h_u, h_tilde_u, w_u, tau1) +
            adaptive_info_nce(h_i, h_tilde_i, w_i, tau1)
        ) / 2.0

        # ── Semantic contrastive loss ─────────────────────────────────────────
        e_u = self.proj(self.user_sem[user_ids])
        e_i = self.proj(self.item_sem[item_ids])
        sem_loss = (
            info_nce(h_u, e_u, self.tau2) +
            info_nce(h_i, e_i, self.tau2)
        ) / 2.0

        return ada_svd_loss, sem_loss
'''

with open(f'{REPO}/semlightgcl.py', 'w') as f:
    f.write(sem_code.strip())

print('✅ semlightgcl.py written to', REPO)

✅ semlightgcl.py written to /kaggle/working/LightGCL


## Cell 3 — Write `precompute_sem.py` (E5 Embedding Extraction)

In [4]:
precompute_code = '''
"""
precompute_sem.py  —  Extract E5-small-v2 semantic embeddings for all items.

Run ONCE before training (CPU is fine, no GPU needed):
    python precompute_sem.py --dataset yelp --data_path ./data
    python precompute_sem.py --dataset amazon --data_path ./data
    python precompute_sem.py --dataset gowalla --data_path ./data

Output: data/{dataset}/item_sem_embeds.pt   shape=(n_items, 384)
        data/{dataset}/user_sem_embeds.pt   shape=(n_users, 384)
"""

import argparse, os, json
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


# ─── Dataset-specific text loading ─────────────────────────────────────────────

def load_item_texts_yelp(data_dir):
    """
    Load business info for Yelp2018.
    Expects data_dir/item_list.txt with format:  org_id  remap_id
    and optionally data_dir/yelp_business.json from the Yelp Open Dataset.
    Falls back to category-only text if full Yelp data not available.
    """
    texts = {}
    # Try loading remap file to get original IDs
    remap_file = os.path.join(data_dir, "item_list.txt")
    business_file = os.path.join(data_dir, "yelp_business.json")

    if os.path.exists(remap_file) and os.path.exists(business_file):
        print("Loading full Yelp business metadata...")
        # Load business info
        biz_info = {}
        with open(business_file) as f:
            for line in f:
                b = json.loads(line)
                cats = b.get("categories") or ""
                biz_info[b["business_id"]] = f"{b["name"]}. Categories: {cats}."
        # Map remap_id -> text
        with open(remap_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    org_id, remap_id = parts[0], int(parts[1])
                    texts[remap_id] = biz_info.get(org_id, f"business {org_id}")
    else:
        print("Full Yelp metadata not found. Using fallback text (item IDs).")
        print("To get better results, download yelp_academic_dataset_business.json")
        print("from https://www.yelp.com/dataset and place it in", data_dir)

    return texts


def load_item_texts_amazon(data_dir):
    """
    Load book metadata for Amazon-Book.
    Expects data_dir/item_list.txt  and  data_dir/amazon_books_meta.jsonl
    Download from: https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/
    """
    texts = {}
    remap_file = os.path.join(data_dir, "item_list.txt")
    meta_file  = os.path.join(data_dir, "amazon_books_meta.jsonl")

    if os.path.exists(remap_file) and os.path.exists(meta_file):
        print("Loading Amazon-Book metadata...")
        meta = {}
        with open(meta_file) as f:
            for line in f:
                item = json.loads(line)
                asin = item.get("asin", "")
                title = item.get("title", "")
                cat   = " ".join(item.get("category", [])[:3])
                desc  = (item.get("description") or [""])[0][:200]
                meta[asin] = f"{title}. Category: {cat}. {desc}"
        with open(remap_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    asin, remap_id = parts[0], int(parts[1])
                    texts[remap_id] = meta.get(asin, f"book {asin}")
    else:
        print("Amazon metadata not found. Using fallback text.")
        print("Download from: https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/")

    return texts


def load_item_texts_gowalla(data_dir):
    """
    Gowalla has minimal text (venue names only). We use location type + name.
    Expects data_dir/item_list.txt  (venue name in column 0 if available).
    """
    texts = {}
    remap_file = os.path.join(data_dir, "item_list.txt")
    if os.path.exists(remap_file):
        with open(remap_file) as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) >= 2:
                    name, remap_id = parts[0], int(parts[1])
                    texts[remap_id] = f"location: {name}"
    return texts


LOADERS = {
    "yelp":    load_item_texts_yelp,
    "amazon":  load_item_texts_amazon,
    "gowalla": load_item_texts_gowalla,
}


# ─── Count items / users from interaction files ─────────────────────────────────
def count_entities(data_dir):
    n_users, n_items = 0, 0
    for split in ["train.txt", "test.txt"]:
        fpath = os.path.join(data_dir, split)
        if not os.path.exists(fpath):
            continue
        with open(fpath) as f:
            for line in f:
                parts = list(map(int, line.strip().split()))
                if not parts:
                    continue
                n_users = max(n_users, parts[0] + 1)
                if len(parts) > 1:
                    n_items = max(n_items, max(parts[1:]) + 1)
    return n_users, n_items


# ─── Build interaction matrix for user mean-pooling ─────────────────────────────
def build_adj(data_dir, n_users, n_items):
    from scipy.sparse import lil_matrix
    adj = lil_matrix((n_users, n_items))
    train_file = os.path.join(data_dir, "train.txt")
    with open(train_file) as f:
        for line in f:
            parts = list(map(int, line.strip().split()))
            if len(parts) < 2:
                continue
            uid = parts[0]
            for iid in parts[1:]:
                adj[uid, iid] = 1
    return adj.tocsr()


# ─── Main ────────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset",   type=str,   default="yelp",
                        choices=["yelp","amazon","gowalla"])
    parser.add_argument("--data_path", type=str,   default="./data")
    parser.add_argument("--model",     type=str,   default="intfloat/e5-small-v2")
    parser.add_argument("--batch",     type=int,   default=512)
    args = parser.parse_args()

    data_dir = os.path.join(args.data_path, args.dataset)
    os.makedirs(data_dir, exist_ok=True)

    # 1. Count entities
    n_users, n_items = count_entities(data_dir)
    print(f"Dataset: {args.dataset}  |  users={n_users}  items={n_items}")

    # 2. Load item texts
    texts = LOADERS[args.dataset](data_dir)

    # 3. Build list of texts in item-ID order (fallback to ID-based text)
    item_strings = [
        f"passage: {texts.get(i, f'item {i}')}"
        for i in range(n_items)
    ]
    text_coverage = sum(1 for i in range(n_items) if i in texts) / max(n_items, 1)
    print(f"Text coverage: {text_coverage*100:.1f}%  "
          f"({'good' if text_coverage > 0.9 else 'low — download metadata for better results'})")

    # 4. Encode with E5-small-v2 (CPU, ~384-dim)
    print(f"Loading {args.model}...")
    encoder = SentenceTransformer(args.model)

    print(f"Encoding {n_items} items (batch={args.batch})...")
    item_embeds = encoder.encode(
        item_strings,
        batch_size=args.batch,
        show_progress_bar=True,
        normalize_embeddings=True,
        device="cpu",
    )
    item_embeds = torch.FloatTensor(item_embeds)

    # 5. Compute user embeddings via mean-pool
    print("Building interaction matrix for user mean-pooling...")
    adj = build_adj(data_dir, n_users, n_items)
    import scipy.sparse as sp
    deg = torch.FloatTensor(np.asarray(adj.sum(1)).flatten()).clamp(min=1).unsqueeze(1)
    user_embeds_np = (adj @ item_embeds.numpy()) / deg.numpy()
    user_embeds = torch.FloatTensor(user_embeds_np)
    # L2-normalize
    user_embeds = torch.nn.functional.normalize(user_embeds, dim=-1)

    # 6. Save
    item_path = os.path.join(data_dir, "item_sem_embeds.pt")
    user_path = os.path.join(data_dir, "user_sem_embeds.pt")
    torch.save(item_embeds, item_path)
    torch.save(user_embeds, user_path)
    print(f"Saved item embeddings: {item_embeds.shape} → {item_path}")
    print(f"Saved user embeddings: {user_embeds.shape} → {user_path}")


if __name__ == "__main__":
    main()
'''

with open(f'{REPO}/precompute_sem.py', 'w') as f:
    f.write(precompute_code.strip())

print('precompute_sem.py written.')

precompute_sem.py written.


## Cell 4 — Write `main_sem.py` (Modified Training Script)

In [5]:
main_sem_code = '''
"""
main_sem.py  —  SemLightGCL training script.

Wraps the official LightGCL training loop and injects:
  - Semantic contrastive loss   (lambda2 * L_cl_sem)
  - Adaptive SVD weighting      (replaces uniform L_cl_svd)

Usage:
    python main_sem.py --dataset yelp   --lambda2 0.1 --gamma 0.5 --tau2 0.2
    python main_sem.py --dataset amazon --lambda2 0.1 --gamma 0.5 --tau2 0.2
    python main_sem.py --dataset gowalla --lambda2 0.05 --gamma 0.5
"""

import argparse, os, sys, time
import torch
import torch.optim as optim
import numpy as np

# ── Make sure the repo root is on the path ──────────────────────────────────
REPO = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, REPO)


def get_args():
    parser = argparse.ArgumentParser("SemLightGCL")
    # ── Inherit all original LightGCL args ──
    parser.add_argument("--dataset",    type=str,   default="yelp",
                        choices=["yelp","amazon","gowalla"])
    parser.add_argument("--data_path",  type=str,   default="./data")
    parser.add_argument("--emb_dim",    type=int,   default=64)
    parser.add_argument("--layers",     type=int,   default=2)
    parser.add_argument("--q",          type=int,   default=5,
                        help="SVD rank")
    parser.add_argument("--lambda1",    type=float, default=0.2,
                        help="Weight for adaptive SVD contrastive loss")
    parser.add_argument("--tau1",       type=float, default=0.2,
                        help="Temperature for SVD contrastive loss")
    parser.add_argument("--lr",         type=float, default=1e-3)
    parser.add_argument("--decay",      type=float, default=1e-4,
                        help="L2 regularization")
    parser.add_argument("--batch_size", type=int,   default=2048)
    parser.add_argument("--epochs",     type=int,   default=200)
    parser.add_argument("--patience",   type=int,   default=10)
    parser.add_argument("--topk",       type=int,   default=20)
    # ── SemLightGCL-specific args ──
    parser.add_argument("--lambda2",    type=float, default=0.1,
                        help="Weight for semantic contrastive loss (NEW)")
    parser.add_argument("--tau2",       type=float, default=0.2,
                        help="Temperature for semantic contrastive loss (NEW)")
    parser.add_argument("--gamma",      type=float, default=0.5,
                        help="Adaptive weighting exponent (NEW): 0=uniform")
    parser.add_argument("--ablation",   type=str,   default="full",
                        choices=["full","no_sem","no_ada","baseline"],
                        help="Ablation mode: full=SemLightGCL, baseline=original LightGCL")
    parser.add_argument("--device",     type=str,   default="cuda")
    return parser.parse_args()


# ─────────────────────────────────────────────────────────────────────────────
#  Data loading (compatible with LightGCL data format)
# ─────────────────────────────────────────────────────────────────────────────

def load_interactions(data_dir):
    """Returns train_dict, test_dict, n_users, n_items."""
    train_dict, test_dict = {}, {}
    n_users = n_items = 0

    for split, d in [("train.txt", train_dict), ("test.txt", test_dict)]:
        fpath = os.path.join(data_dir, split)
        if not os.path.exists(fpath):
            continue
        with open(fpath) as f:
            for line in f:
                parts = list(map(int, line.strip().split()))
                if not parts:
                    continue
                uid = parts[0]
                items = parts[1:]
                d[uid] = items
                n_users = max(n_users, uid + 1)
                if items:
                    n_items = max(n_items, max(items) + 1)

    return train_dict, test_dict, n_users, n_items


def build_sparse_adj(train_dict, n_users, n_items):
    from scipy.sparse import lil_matrix
    adj = lil_matrix((n_users, n_items))
    for uid, items in train_dict.items():
        for iid in items:
            adj[uid, iid] = 1.0
    return adj.tocsr()


# ─────────────────────────────────────────────────────────────────────────────
#  LightGCL model (self-contained — matches paper architecture)
# ─────────────────────────────────────────────────────────────────────────────

import torch.nn as nn
import torch.nn.functional as F
from torch_sparse import SparseTensor


def to_torch_sparse(adj, device):
    """Convert scipy CSR to torch SparseTensor (or fall back to dense)."""
    adj_coo = adj.tocoo()
    row = torch.LongTensor(adj_coo.row).to(device)
    col = torch.LongTensor(adj_coo.col).to(device)
    val = torch.FloatTensor(adj_coo.data).to(device)
    return torch.sparse_coo_tensor(
        torch.stack([row, col]), val, adj_coo.shape
    ).to(device)


def svd_augment(adj_scipy, rank, device):
    """
    Compute truncated SVD and return the sparse augmented adjacency.
    Returns R_tilde as a dense tensor (manageable for these dataset sizes).
    """
    import scipy.sparse.linalg as la
    print(f"  SVD at rank={rank}...")
    U, S, Vt = la.svds(adj_scipy.astype(np.float32), k=rank)
    U  = torch.FloatTensor(U).to(device)
    S  = torch.FloatTensor(S).to(device)
    Vt = torch.FloatTensor(Vt).to(device)
    # R_tilde = U diag(S) Vt  — will be used for GNN propagation
    return U, S, Vt


class LightGCN_Layer(nn.Module):
    """Single LightGCN propagation layer."""
    def forward(self, embed, adj_t):
        # adj_t: (n_items+n_users, n_users+n_items) normalized bipartite
        return torch.sparse.mm(adj_t, embed)


class LightGCL(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers, adj, U_svd, S_svd, Vt_svd, device):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.emb_dim = emb_dim
        self.n_layers = n_layers
        self.device   = device

        # Embeddings
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

        # SVD factors (frozen buffers)
        self.register_buffer("U_svd",  U_svd)   # (n_users, rank)
        self.register_buffer("S_svd",  S_svd)   # (rank,)
        self.register_buffer("Vt_svd", Vt_svd)  # (rank, n_items)

        # Normalized bipartite adjacency for GNN propagation on R
        self.adj_norm = self._normalize_adj(adj)

    def _normalize_adj(self, adj):
        """Symmetric D^{-1/2} A D^{-1/2} normalization."""
        import scipy.sparse as sp
        n = self.n_users + self.n_items
        # Build bipartite adjacency
        zeros_uu = sp.csr_matrix((self.n_users, self.n_users))
        zeros_ii = sp.csr_matrix((self.n_items, self.n_items))
        A = sp.bmat([[zeros_uu, adj], [adj.T, zeros_ii]]).tocsr()
        # Degree normalization
        d = np.asarray(A.sum(1)).flatten() + 1e-8
        D_inv_sqrt = sp.diags(1.0 / np.sqrt(d))
        A_norm = D_inv_sqrt @ A @ D_inv_sqrt
        coo = A_norm.tocoo()
        idx = torch.LongTensor(np.stack([coo.row, coo.col]))
        val = torch.FloatTensor(coo.data)
        return torch.sparse_coo_tensor(idx, val, (n, n)).to(self.device)

    def _gnn_forward(self, adj_norm):
        """L-layer LightGCN propagation. Returns mean-pooled embeddings."""
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(adj_norm, all_emb)
            embs.append(all_emb)
        # Mean pooling over layers
        all_emb = torch.stack(embs, dim=1).mean(dim=1)
        h_u = all_emb[:self.n_users]
        h_i = all_emb[self.n_users:]
        return h_u, h_i

    def _svd_adj_norm(self):
        """
        Build a normalized bipartite adjacency from R_tilde = U_svd diag(S) Vt_svd.
        Returns a function that performs one propagation step.
        For efficiency we propagate directly through the SVD factors.
        """
        # We propagate: h = U_svd diag(S) (Vt_svd @ item_emb)  for users
        # and           h = Vt_svd.T diag(S) (U_svd.T @ user_emb)  for items
        # (simplified single-layer SVD propagation)
        return None  # signal to use SVD propagation

    def forward(self):
        """Returns h_u, h_i (main graph) and h_tilde_u, h_tilde_i (SVD graph)."""
        # Main graph embeddings
        h_u, h_i = self._gnn_forward(self.adj_norm)

        # SVD graph embeddings (single LightGCN-style propagation via SVD factors)
        # User embedding update: aggregate from SVD-approximated item neighbors
        # E_u_tilde = U_svd @ diag(S_svd) @ (Vt_svd @ E_i)
        E_i_base = self.item_emb.weight           # (n_items, d)
        E_u_base = self.user_emb.weight           # (n_users, d)
        # User SVD update:
        tmp = self.Vt_svd @ E_i_base              # (rank, d)
        h_tilde_u = self.U_svd @ (self.S_svd.unsqueeze(1) * tmp) + E_u_base
        # Item SVD update:
        tmp2 = self.U_svd.T @ E_u_base            # (rank, d)
        h_tilde_i = self.Vt_svd.T @ (self.S_svd.unsqueeze(1) * tmp2) + E_i_base

        # L2-normalize SVD embeddings
        h_tilde_u = F.normalize(h_tilde_u, dim=-1)
        h_tilde_i = F.normalize(h_tilde_i, dim=-1)

        return h_u, h_i, h_tilde_u, h_tilde_i

    def predict(self, user_ids):
        h_u, h_i, _, _ = self.forward()
        scores = h_u[user_ids] @ h_i.T
        return scores


# ─────────────────────────────────────────────────────────────────────────────
#  BPR Loss
# ─────────────────────────────────────────────────────────────────────────────

def bpr_loss(h_u, h_i, user_ids, pos_ids, neg_ids):
    u_emb  = h_u[user_ids]
    pos_emb = h_i[pos_ids]
    neg_emb = h_i[neg_ids]
    pos_score = (u_emb * pos_emb).sum(dim=-1)
    neg_score = (u_emb * neg_emb).sum(dim=-1)
    loss = -F.logsigmoid(pos_score - neg_score).mean()
    # L2 regularization on embeddings
    reg = (u_emb.norm(2).pow(2) + pos_emb.norm(2).pow(2) + neg_emb.norm(2).pow(2)) / (2 * len(user_ids))
    return loss, reg


# ─────────────────────────────────────────────────────────────────────────────
#  Evaluation
# ─────────────────────────────────────────────────────────────────────────────

def evaluate(model, train_dict, test_dict, n_users, n_items, device, topk=20, batch=512):
    model.eval()
    h_u, h_i, _, _ = model.forward()
    h_u = h_u.detach()
    h_i = h_i.detach()

    recall_list, ndcg_list = [], []
    test_users = [u for u in test_dict if test_dict[u]]

    for start in range(0, len(test_users), batch):
        batch_users = test_users[start:start+batch]
        uid_tensor  = torch.LongTensor(batch_users).to(device)
        scores = h_u[uid_tensor] @ h_i.T   # (batch, n_items)

        # Mask training items
        for i, uid in enumerate(batch_users):
            train_items = train_dict.get(uid, [])
            if train_items:
                scores[i, train_items] = -1e9

        _, top_items = scores.topk(topk, dim=1)
        top_items = top_items.cpu().numpy()

        for i, uid in enumerate(batch_users):
            gt = set(test_dict[uid])
            preds = top_items[i]
            hits = [1 if p in gt else 0 for p in preds]
            recall = sum(hits) / max(len(gt), 1)
            recall_list.append(recall)
            # NDCG
            dcg = sum(h / np.log2(r + 2) for r, h in enumerate(hits))
            ideal = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), topk)))
            ndcg_list.append(dcg / max(ideal, 1e-8))

    model.train()
    return float(np.mean(recall_list)), float(np.mean(ndcg_list))


# ─────────────────────────────────────────────────────────────────────────────
#  Main training loop
# ─────────────────────────────────────────────────────────────────────────────

def sample_negatives(train_dict, n_items, user_ids):
    neg_ids = []
    for uid in user_ids:
        train_set = set(train_dict.get(uid.item(), []))
        neg = np.random.randint(0, n_items)
        while neg in train_set:
            neg = np.random.randint(0, n_items)
        neg_ids.append(neg)
    return torch.LongTensor(neg_ids)


def train(args):
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"SemLightGCL  |  dataset={args.dataset}  |  ablation={args.ablation}")
    print(f"lambda1={args.lambda1}  lambda2={args.lambda2}  gamma={args.gamma}  tau2={args.tau2}")
    print(f"{'='*60}\n")

    # 1. Load data
    data_dir = os.path.join(args.data_path, args.dataset)
    print("Loading interactions...")
    train_dict, test_dict, n_users, n_items = load_interactions(data_dir)
    print(f"  users={n_users}, items={n_items}, train_interactions={sum(len(v) for v in train_dict.values())}")

    adj = build_sparse_adj(train_dict, n_users, n_items)

    # 2. Build LightGCL model with SVD
    U_svd, S_svd, Vt_svd = svd_augment(adj, args.q, device)
    model = LightGCL(n_users, n_items, args.emb_dim, args.layers,
                     adj, U_svd, S_svd, Vt_svd, device).to(device)

    # 3. SemExtension (only if not pure baseline)
    sem_ext = None
    if args.ablation != "baseline":
        item_sem_path = os.path.join(data_dir, "item_sem_embeds.pt")
        if not os.path.exists(item_sem_path):
            print(f"\n Semantic embeddings not found at {item_sem_path}")
            print("   Run:  python precompute_sem.py --dataset", args.dataset)
            print("   Falling back to baseline mode.\n")
            args.ablation = "baseline"
        else:
            from semlightgcl import SemExtension
            item_sem = torch.load(item_sem_path, map_location="cpu")
            sem_ext  = SemExtension(
                item_sem_embeds = item_sem,
                adj             = adj,
                emb_dim         = args.emb_dim,
                lambda2         = args.lambda2,
                tau2            = args.tau2,
                gamma           = args.gamma if args.ablation != "no_ada" else 0.0,
                device          = str(device),
            ).to(device)
            print(f"  SemExtension loaded. Projection params: "
                  f"{sum(p.numel() for p in sem_ext.parameters()):,}")

    # 4. Optimizer
    params = list(model.parameters())
    if sem_ext is not None:
        params += list(sem_ext.proj.parameters())
    optimizer = optim.Adam(params, lr=args.lr, weight_decay=args.decay)

    # 5. Build training pairs
    all_users, all_pos = [], []
    for uid, items in train_dict.items():
        for iid in items:
            all_users.append(uid)
            all_pos.append(iid)
    all_users = torch.LongTensor(all_users)
    all_pos   = torch.LongTensor(all_pos)

    # 6. Training loop
    best_recall, best_ndcg, no_improve = 0.0, 0.0, 0
    results_log = []

    from semlightgcl import info_nce, adaptive_info_nce
    import math

    for epoch in range(1, args.epochs + 1):
        model.train()
        t0 = time.time()
        # Shuffle
        perm = torch.randperm(len(all_users))
        total_loss = 0.0
        n_batches  = 0

        for start in range(0, len(all_users), args.batch_size):
            idx      = perm[start:start + args.batch_size]
            uid_b    = all_users[idx].to(device)
            pos_b    = all_pos[idx].to(device)
            neg_b    = sample_negatives(train_dict, n_items, all_users[idx]).to(device)

            # Forward
            h_u, h_i, h_tilde_u, h_tilde_i = model.forward()

            # BPR loss
            l_bpr, l_reg = bpr_loss(h_u, h_i, uid_b, pos_b, neg_b)

            # Unique users and items for contrastive losses
            uniq_u = uid_b.unique()
            uniq_i = pos_b.unique()

            if args.ablation == "baseline":
                # Original LightGCL: uniform InfoNCE on SVD views
                l_cl = (
                    info_nce(h_u[uniq_u], h_tilde_u[uniq_u], args.tau1) +
                    info_nce(h_i[uniq_i], h_tilde_i[uniq_i], args.tau1)
                ) / 2.0
                loss = l_bpr + args.decay * l_reg + args.lambda1 * l_cl

            else:
                # SemLightGCL: adaptive SVD CL + semantic CL
                ada_svd_loss, sem_loss = sem_ext(
                    uniq_u, uniq_i,
                    h_u[uniq_u], h_i[uniq_i],
                    h_tilde_u[uniq_u], h_tilde_i[uniq_i],
                    tau1=args.tau1
                )
                if args.ablation == "no_sem":
                    sem_loss = torch.tensor(0.0, device=device)
                loss = (l_bpr + args.decay * l_reg
                        + args.lambda1 * ada_svd_loss
                        + args.lambda2 * sem_loss)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        avg_loss = total_loss / max(n_batches, 1)
        elapsed  = time.time() - t0

        # Evaluate every epoch
        recall, ndcg = evaluate(model, train_dict, test_dict, n_users, n_items,
                                device, topk=args.topk)
        results_log.append({"epoch": epoch, "loss": avg_loss, "recall": recall, "ndcg": ndcg})

        print(f"Epoch {epoch:3d}/{args.epochs}  "
              f"loss={avg_loss:.4f}  "
              f"R@{args.topk}={recall:.4f}  "
              f"N@{args.topk}={ndcg:.4f}  "
              f"({elapsed:.0f}s)")

        if recall > best_recall:
            best_recall, best_ndcg = recall, ndcg
            no_improve = 0
            # Save checkpoint
            torch.save(model.state_dict(),
                       f"{args.dataset}_{args.ablation}_best_model.pt")
            print(f"New best  R@{args.topk}={best_recall:.4f}  N@{args.topk}={best_ndcg:.4f}")
        else:
            no_improve += 1
            if no_improve >= args.patience:
                print(f"  Early stopping after {epoch} epochs.")
                break

    print(f"\n{'='*60}")
    print(f"FINAL  dataset={args.dataset}  ablation={args.ablation}")
    print(f"Best  Recall@{args.topk} = {best_recall:.4f}")
    print(f"Best  NDCG@{args.topk}   = {best_ndcg:.4f}")
    print(f"{'='*60}\n")

    return best_recall, best_ndcg, results_log


if __name__ == "__main__":
    args = get_args()
    train(args)
'''

with open(f'{REPO}/main_sem.py', 'w') as f:
    f.write(main_sem_code.strip())

print('main_sem.py written.')

main_sem.py written.


## Cell 5 — Pre-compute E5 Embeddings

In [6]:
import os, sys, subprocess, zipfile

REPO = '/kaggle/working/LightGCL'
DATA = f'{REPO}/data'

# ── Step 1: Re-clone if repo was wiped by Kaggle session reset ────────────────
if not os.path.exists(REPO):
    print("Repo missing — re-cloning...")
    subprocess.run(['git', 'clone',
                    'https://github.com/HKUDS/LightGCL.git', REPO],
                   check=True)
    # Re-write our custom files
    print("Re-run Cells 2, 3, 4 to restore semlightgcl.py / main_sem.py")
else:
    print(f"Repo exists: {REPO}")

sys.path.insert(0, REPO)
os.chdir(REPO)

# ── Step 2: Extract any zipped datasets ──────────────────────────────────────
for fname in os.listdir(DATA):
    if fname.endswith('.zip'):
        zip_path  = os.path.join(DATA, fname)
        out_folder = os.path.join(DATA, fname.replace('.zip', ''))
        if not os.path.exists(out_folder):
            print(f"Extracting {fname}...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(DATA)
            print(f"Extracted → {out_folder}")
        else:
            print(f"Already extracted: {out_folder}")

# ── Step 3: Show exactly what is inside each dataset folder ──────────────────
print(f"\n{'='*55}")
print("Full data directory tree:")
for dataset_dir in sorted(os.listdir(DATA)):
    full = os.path.join(DATA, dataset_dir)
    if os.path.isdir(full):
        files = os.listdir(full)
        print(f"\n  📁 data/{dataset_dir}/")
        for f in files:
            fp   = os.path.join(full, f)
            size = os.path.getsize(fp) // 1024
            print(f"     {f:<30} {size:>6} KB")
print(f"{'='*55}\n")

# ── Step 4: Run fixed precompute (handles trnMat.pkl correctly) ───────────────
fixed_precompute = '''
import argparse, os, pickle
import numpy as np
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import scipy.sparse as sp


def load_adj(data_dir):
    """
    Auto-detect LightGCL data format.
    Tries (in order):
      trnMat.pkl   ← HKUDS standard (camelCase)
      trn_mat.pkl  ← some forks (snake_case)
      train.txt    ← plain text fallback
    """
    candidates = [
        ("trnMat.pkl",   "pkl"),
        ("trn_mat.pkl",  "pkl"),
        ("trnMat.npz",   "npz"),
        ("trn_mat.npz",  "npz"),
        ("train.txt",    "txt"),
    ]
    for fname, fmt in candidates:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            continue
        print(f"  Found: {fname}  (format={fmt})")
        if fmt == "pkl":
            with open(fpath, "rb") as f:
                adj = pickle.load(f)
        elif fmt == "npz":
            adj = sp.load_npz(fpath)
        else:
            # plain text: user_id item_id1 item_id2 ...
            rows, cols = [], []
            n_users = n_items = 0
            with open(fpath) as f:
                for line in f:
                    parts = list(map(int, line.strip().split()))
                    if not parts: continue
                    uid, items = parts[0], parts[1:]
                    n_users = max(n_users, uid + 1)
                    for iid in items:
                        n_items = max(n_items, iid + 1)
                        rows.append(uid); cols.append(iid)
            adj = sp.csr_matrix(
                (np.ones(len(rows)), (rows, cols)),
                shape=(n_users, n_items)
            )
            n_users, n_items = adj.shape
            print(f"  Shape: {n_users} users x {n_items} items  nnz={adj.nnz}")
            return adj.tocsr(), n_users, n_items

        if not sp.issparse(adj):
            adj = sp.csr_matrix(adj)
        else:
            adj = adj.tocsr()
        n_users, n_items = adj.shape
        print(f"  Shape: {n_users} users x {n_items} items  nnz={adj.nnz}")
        return adj, n_users, n_items

    raise FileNotFoundError(
        f"No data file found in {data_dir}\\n"
        f"Files present: {os.listdir(data_dir)}"
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset",   type=str, default="yelp",
                        choices=["yelp","amazon","gowalla"])
    parser.add_argument("--data_path", type=str, default="./data")
    parser.add_argument("--model",     type=str, default="intfloat/e5-small-v2")
    parser.add_argument("--batch",     type=int, default=256)
    args = parser.parse_args()

    data_dir = os.path.join(args.data_path, args.dataset)
    print(f"\\nDataset : {args.dataset}")
    print(f"Files   : {os.listdir(data_dir)}")

    # Load adj matrix
    adj, n_users, n_items = load_adj(data_dir)

    # Build fallback item strings (no external text needed)
    print(f"Building item text strings for {n_items} items...")
    item_strings = [f"passage: item {i}" for i in range(n_items)]

    # Encode with E5-small-v2
    print(f"Loading {args.model}...")
    encoder = SentenceTransformer(args.model)
    print(f"Encoding {n_items} items...")
    item_embeds = encoder.encode(
        item_strings, batch_size=args.batch,
        show_progress_bar=True,
        normalize_embeddings=True,
        device="cpu",
    )
    item_embeds = torch.FloatTensor(item_embeds)

    # User embeddings via mean-pool
    print("Computing user embeddings...")
    deg = torch.FloatTensor(
        np.asarray(adj.sum(1)).flatten()
    ).clamp(min=1).unsqueeze(1)
    user_np     = (adj @ item_embeds.numpy()) / deg.numpy()
    user_embeds = F.normalize(torch.FloatTensor(user_np), dim=-1)

    # Save
    torch.save(item_embeds, os.path.join(data_dir, "item_sem_embeds.pt"))
    torch.save(user_embeds, os.path.join(data_dir, "user_sem_embeds.pt"))
    print(f"\\n item_sem_embeds.pt  {item_embeds.shape}")
    print(f" user_sem_embeds.pt  {user_embeds.shape}")


if __name__ == "__main__":
    main()
'''

with open(f'{REPO}/precompute_sem.py', 'w') as f:
    f.write(fixed_precompute.strip())
print("File precompute_sem.py patched")

# ── Step 5: Run precompute for available datasets ─────────────────────────────
available = [
    d for d in ['yelp', 'gowalla', 'amazon']
    if os.path.isdir(os.path.join(DATA, d))
]
print(f"\nAvailable datasets: {available}")

for dataset in available:
    embed_path = os.path.join(DATA, dataset, 'item_sem_embeds.pt')
    if os.path.exists(embed_path):
        t = torch.load(embed_path, map_location='cpu')
        print(f"[{dataset}] Already done: {t.shape} — skipping")
        continue
    print(f"\n--- Pre-computing: {dataset} ---")
    subprocess.run(
        [sys.executable, 'precompute_sem.py',
         '--dataset', dataset,
         '--data_path', './data',
         '--batch', '256'],
        check=False
    )

Repo exists: /kaggle/working/LightGCL
Already extracted: /kaggle/working/LightGCL/data/amazon
Already extracted: /kaggle/working/LightGCL/data/ml10m
Already extracted: /kaggle/working/LightGCL/data/tmall

Full data directory tree:

  📁 data/amazon/
     tstMat.pkl                      10001 KB
     user_sem_embeds.pt             117868 KB
     trnMat.pkl                      35002 KB
     item_sem_embeds.pt             116703 KB

  📁 data/gowalla/
     tstMat.pkl                       2035 KB
     user_sem_embeds.pt              76233 KB
     trnMat.pkl                      18319 KB
     item_sem_embeds.pt              86161 KB

  📁 data/ml-10m/
     ml-10M100K                          4 KB
     tstMat.pkl                      12217 KB
     user_sem_embeds.pt              99024 KB
     trnMat.pkl                      46544 KB
     item_sem_embeds.pt              11692 KB
     ml-10m.zip                      64029 KB

  📁 data/ml10m/
     tstMat.pkl                      31246 KB
     tr

## Cell 6 — Install torch-sparse (needed for sparse GNN ops)

In [7]:
# torch-sparse must match your PyTorch and CUDA version
import torch
pv = torch.__version__.split('+')[0]   # e.g. '2.1.0'
cu = torch.version.cuda               # e.g. '11.8'
cu_tag = 'cu' + cu.replace('.','')[:3]  # e.g. 'cu118'
print(f'PyTorch {pv}, CUDA {cu}  →  tag={cu_tag}')

# Install matching torch-sparse
!pip install torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{pv}+{cu_tag}.html -q
print('\n torch-scatter and torch-sparse installed.')

PyTorch 2.10.0, CUDA 12.8  →  tag=cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 85.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 48.9 MB/s eta 0:00:00

 torch-scatter and torch-sparse installed.


## Cell 7 — Run Training

In [8]:
# ── Ghi thẳng main_sem.py ─────────────────────────────────────────────────────
path = '/kaggle/working/LightGCL/main_sem.py'

code = r"""
import argparse, os, sys, time, pickle, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import scipy.sparse as sp

REPO = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, REPO)


def get_args():
    p = argparse.ArgumentParser("SemLightGCL")
    p.add_argument("--dataset",    default="yelp", choices=["yelp","amazon","gowalla"])
    p.add_argument("--data_path",  default="./data")
    p.add_argument("--emb_dim",    type=int,   default=64)
    p.add_argument("--layers",     type=int,   default=2)
    p.add_argument("--q",          type=int,   default=5)
    p.add_argument("--lambda1",    type=float, default=0.2)
    p.add_argument("--lambda2",    type=float, default=0.1)
    p.add_argument("--tau1",       type=float, default=0.2)
    p.add_argument("--tau2",       type=float, default=0.2)
    p.add_argument("--gamma",      type=float, default=0.5)
    p.add_argument("--lr",         type=float, default=1e-3)
    p.add_argument("--decay",      type=float, default=1e-4)
    p.add_argument("--batch_size", type=int,   default=2048)
    p.add_argument("--epochs",     type=int,   default=200)
    p.add_argument("--patience",   type=int,   default=10)
    p.add_argument("--topk",       type=int,   default=20)
    p.add_argument("--ablation",   default="full", choices=["full","no_sem","no_ada","baseline"])
    p.add_argument("--device",     default="cuda")
    return p.parse_args()


def load_data(data_dir):
    with open(os.path.join(data_dir, "trnMat.pkl"), "rb") as f:
        trn = pickle.load(f).tocsr()
    with open(os.path.join(data_dir, "tstMat.pkl"), "rb") as f:
        tst = pickle.load(f).tocsr()
    n_users, n_items = trn.shape
    train_dict, test_dict = {}, {}
    for u in range(n_users):
        ti = trn.getrow(u).indices.tolist()
        if ti: train_dict[u] = ti
        te = tst.getrow(u).indices.tolist()
        if te: test_dict[u]  = te
    print(f"  users={n_users}  items={n_items}  train_nnz={trn.nnz}  test_nnz={tst.nnz}")
    return trn, tst, train_dict, test_dict, n_users, n_items


def build_norm_adj(trn_csr, n_users, n_items, device):
    n = n_users + n_items
    zeros_uu = sp.csr_matrix((n_users, n_users))
    zeros_ii = sp.csr_matrix((n_items, n_items))
    A = sp.bmat([[zeros_uu, trn_csr],
                 [trn_csr.T, zeros_ii]], format="csr")
    d   = np.asarray(A.sum(1)).flatten() + 1e-8
    D05 = sp.diags(1.0 / np.sqrt(d))
    A_norm = (D05 @ A @ D05).tocoo()
    idx = torch.LongTensor(np.stack([A_norm.row, A_norm.col]))
    val = torch.FloatTensor(A_norm.data)
    return torch.sparse_coo_tensor(idx, val, (n, n)).to(device)


def svd_augment(trn_csr, rank, device):
    import scipy.sparse.linalg as la
    print(f"  Computing SVD at rank={rank}...")
    U, S, Vt = la.svds(trn_csr.astype(np.float32), k=rank)
    return (torch.FloatTensor(U.copy()).to(device),
            torch.FloatTensor(S.copy()).to(device),
            torch.FloatTensor(Vt.copy()).to(device))


class LightGCL(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers,
                 adj_norm, U_svd, S_svd, Vt_svd):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        self.adj_norm = adj_norm
        self.register_buffer("U_svd",  U_svd)
        self.register_buffer("S_svd",  S_svd)
        self.register_buffer("Vt_svd", Vt_svd)

    def forward(self):
        x    = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        outs = [x]
        for _ in range(self.n_layers):
            x  = torch.sparse.mm(self.adj_norm, x)
            outs.append(x)
        x   = torch.stack(outs, dim=1).mean(dim=1)
        h_u = x[:self.n_users]
        h_i = x[self.n_users:]
        tmp       = self.Vt_svd @ self.item_emb.weight
        h_tilde_u = F.normalize(self.U_svd @ (self.S_svd.unsqueeze(1) * tmp) + self.user_emb.weight, dim=-1)
        tmp2      = self.U_svd.T @ self.user_emb.weight
        h_tilde_i = F.normalize(self.Vt_svd.T @ (self.S_svd.unsqueeze(1) * tmp2) + self.item_emb.weight, dim=-1)
        return h_u, h_i, h_tilde_u, h_tilde_i


def bpr(h_u, h_i, uids, pos, neg, decay):
    u  = h_u[uids]; pi = h_i[pos]; ni = h_i[neg]
    loss = -F.logsigmoid((u*pi).sum(-1) - (u*ni).sum(-1)).mean()
    reg  = (u.norm(2).pow(2) + pi.norm(2).pow(2) + ni.norm(2).pow(2)) / (2 * len(uids))
    return loss + decay * reg


def info_nce(a, b, temp):
    a = F.normalize(a, dim=-1); b = F.normalize(b, dim=-1)
    N = a.size(0)
    s = a @ b.T / temp
    l = torch.arange(N, device=a.device)
    return (F.cross_entropy(s, l) + F.cross_entropy(s.T, l)) / 2


def adaptive_weights(trn_csr, gamma, device):
    u_deg = torch.FloatTensor(np.asarray(trn_csr.sum(1)).flatten())
    i_deg = torch.FloatTensor(np.asarray(trn_csr.sum(0)).flatten())
    def wfn(d):
        w = 1.0 / torch.pow(torch.log(d + math.e), gamma)
        return (w / w.mean()).to(device)
    return wfn(u_deg), wfn(i_deg)


@torch.no_grad()
def evaluate(model, train_dict, test_dict, n_items, device, topk=20):
    model.eval()
    h_u, h_i, _, _ = model.forward()
    test_users = [u for u in test_dict if test_dict[u]]
    recalls, ndcgs = [], []
    for start in range(0, len(test_users), 512):
        batch  = test_users[start:start+512]
        uids   = torch.LongTensor(batch).to(device)
        scores = (h_u[uids] @ h_i.T).cpu().numpy()
        for i, uid in enumerate(batch):
            for iid in train_dict.get(uid, []):
                scores[i, iid] = -1e9
            top  = np.argpartition(scores[i], -topk)[-topk:]
            top  = top[np.argsort(-scores[i, top])]
            gt   = set(test_dict[uid])
            hits = [1 if p in gt else 0 for p in top]
            recalls.append(sum(hits) / max(len(gt), 1))
            dcg   = sum(h / math.log2(r+2) for r, h in enumerate(hits))
            ideal = sum(1/math.log2(r+2) for r in range(min(len(gt), topk)))
            ndcgs.append(dcg / max(ideal, 1e-8))
    model.train()
    return float(np.mean(recalls)), float(np.mean(ndcgs))


def train():
    args   = get_args()
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    print("=" * 60)
    print(f"SemLightGCL | {args.dataset} | ablation={args.ablation} | {device}")
    print("=" * 60)

    data_dir = os.path.join(args.data_path, args.dataset)
    trn_csr, tst_csr, train_dict, test_dict, n_users, n_items = load_data(data_dir)

    adj_norm              = build_norm_adj(trn_csr, n_users, n_items, device)
    U_svd, S_svd, Vt_svd = svd_augment(trn_csr, args.q, device)
    u_w, i_w              = adaptive_weights(trn_csr, args.gamma, device)

    model = LightGCL(n_users, n_items, args.emb_dim, args.layers,
                     adj_norm, U_svd, S_svd, Vt_svd).to(device)

    sem_ext = None
    if args.ablation != "baseline":
        item_path = os.path.join(data_dir, "item_sem_embeds.pt")
        if os.path.exists(item_path):
            from semlightgcl import SemExtension
            item_sem = torch.load(item_path, map_location="cpu")
            sem_ext  = SemExtension(
                item_sem_embeds=item_sem, adj=trn_csr,
                emb_dim=args.emb_dim,
                lambda2=args.lambda2, tau2=args.tau2,
                gamma=args.gamma if args.ablation != "no_ada" else 0.0,
                device=str(device),
            ).to(device)
            proj_params = sum(p.numel() for p in sem_ext.proj.parameters())
            print(f"  SemExtension ready  proj_params={proj_params}")
        else:
            print("  item_sem_embeds.pt not found — falling back to baseline")
            args.ablation = "baseline"

    params    = list(model.parameters())
    if sem_ext: params += list(sem_ext.proj.parameters())
    optimizer = optim.Adam(params, lr=args.lr)

    all_u = np.array([u  for u, items in train_dict.items() for _ in items])
    all_i = np.array([it for u, items in train_dict.items() for it in items])
    print(f"  Training pairs: {len(all_u)}")

    best_r = best_n = 0.0
    patience = 0

    for epoch in range(1, args.epochs + 1):
        model.train()
        t0      = time.time()
        perm    = np.random.permutation(len(all_u))
        ep_loss = 0.0
        n_batch = 0

        for s in range(0, len(all_u), args.batch_size):
            idx  = perm[s:s + args.batch_size]
            uids = torch.LongTensor(all_u[idx]).to(device)
            pos  = torch.LongTensor(all_i[idx]).to(device)
            neg  = torch.LongTensor(np.random.randint(0, n_items, len(idx))).to(device)

            h_u, h_i, h_tu, h_ti = model.forward()
            loss = bpr(h_u, h_i, uids, pos, neg, args.decay)
            uu = uids.unique()
            ui = pos.unique()

            if args.ablation == "baseline":
                cl   = (info_nce(h_u[uu], h_tu[uu], args.tau1) + info_nce(h_i[ui], h_ti[ui], args.tau1)) / 2
                loss = loss + args.lambda1 * cl
            else:
                ada_svd, sem = sem_ext(uu, ui, h_u[uu], h_i[ui], h_tu[uu], h_ti[ui], tau1=args.tau1)
                if args.ablation == "no_sem":
                    sem = torch.tensor(0., device=device)
                loss = loss + args.lambda1 * ada_svd + args.lambda2 * sem

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
            n_batch += 1

        r, n    = evaluate(model, train_dict, test_dict, n_items, device, args.topk)
        elapsed = int(time.time() - t0)
        print(f"Ep {epoch:3d}  loss={ep_loss/n_batch:.4f}  R@{args.topk}={r:.4f}  N@{args.topk}={n:.4f}  ({elapsed}s)")

        if r > best_r:
            best_r, best_n = r, n
            patience = 0
            torch.save(model.state_dict(), f"{args.dataset}_{args.ablation}_best.pt")
            print(f"  New best  R@{args.topk}={best_r:.4f}  N@{args.topk}={best_n:.4f}")
        else:
            patience += 1
            if patience >= args.patience:
                print(f"  Early stop at epoch {epoch}")
                break

    print(f"FINAL {args.dataset} [{args.ablation}]  R@{args.topk}={best_r:.4f}  N@{args.topk}={best_n:.4f}")
    return best_r, best_n


if __name__ == "__main__":
    train()
""".strip()

with open(path, 'w') as f:
    f.write(code)

import py_compile
try:
    py_compile.compile(path, doraise=True)
    print('main_sem.py written cleanly — run Cell 7 now')
except py_compile.PyCompileError as e:
    print('Error: ', e)

main_sem.py written cleanly — run Cell 7 now


In [9]:
# ── Patch: fix negative-stride SVD arrays in main_sem.py ─────────────────────
path = '/kaggle/working/LightGCL/main_sem.py'

with open(path, 'r') as f:
    code = f.read()

# The one broken line — add .copy() to each SVD array before torch conversion
old = (
    "    U, S, Vt = la.svds(trn_csr.astype(np.float32), k=rank)\n"
    "    return (torch.FloatTensor(U).to(device),\n"
    "            torch.FloatTensor(S).to(device),\n"
    "            torch.FloatTensor(Vt).to(device))"
)
new = (
    "    U, S, Vt = la.svds(trn_csr.astype(np.float32), k=rank)\n"
    "    return (torch.FloatTensor(U.copy()).to(device),\n"
    "            torch.FloatTensor(S.copy()).to(device),\n"
    "            torch.FloatTensor(Vt.copy()).to(device))"
)

if old in code:
    code = code.replace(old, new)
    with open(path, 'w') as f:
        f.write(code)
    print('SVD stride fix applied — run Cell 7 now')
elif 'U.copy()' in code:
    print('Already patched — run Cell 7 directly')
else:
    print('Pattern not found — printing lines 70-78 for inspection:')
    for i, line in enumerate(code.splitlines()[68:78], start=69):
        print(f'  {i}: {repr(line)}')

Already patched — run Cell 7 directly


In [10]:
import subprocess, sys, os, re
REPO = '/kaggle/working/LightGCL'
os.chdir(REPO)

CONFIG_BASE = {
    'data_path':  './data',
    'emb_dim':    64,
    'layers':     2,
    'q':          5,
    'lambda1':    0.2,
    'lambda2':    0.01,
    'tau1':       0.2,
    'tau2':       0.2,
    'gamma':      0.5,
    'lr':         '1e-3',
    'decay':      '1e-4',
    'batch_size': 2048,
    'epochs':     1000,
    'patience':   30,
    'topk':       20,
}

DATASETS  = ['yelp', 'gowalla', 'amazon', 'ml-10m', 'tmall']
ABLATIONS = ['full']
results_store = {}

for dataset in DATASETS:
    for ablation in ABLATIONS:
        print(f'\n{"="*60}')
        print(f'Dataset: {dataset}  |  Ablation: {ablation}')
        print(f'{"="*60}')
        cmd = [sys.executable, 'main_sem.py',
               f'--dataset={dataset}', f'--ablation={ablation}']
        for k, v in CONFIG_BASE.items():
            cmd += [f'--{k}={v}']
        proc = subprocess.run(cmd, capture_output=True, text=True)
        print(proc.stdout)
        if proc.returncode != 0:
            print(f'{dataset}/{ablation} failed'); print(proc.stderr); continue
        match = re.search(r'FINAL \w+ \[\w+\]\s+R@20=([\d.]+)\s+N@20=([\d.]+)', proc.stdout)
        if match:
            r, n = float(match.group(1)), float(match.group(2))
            results_store[(dataset, ablation)] = (r, n)
            print(f'{dataset}/{ablation}  R@20={r:.4f}  N@20={n:.4f}')

print('\nAll runs complete.')


Dataset: yelp  |  Ablation: full
SemLightGCL | yelp | ablation=full | cuda
  users=29601  items=24734  train_nnz=1069128  test_nnz=305466
  Computing SVD at rank=5...
  SemExtension ready  proj_params=24640
  Training pairs: 1069128
Ep   1  loss=1.3871  R@20=0.0016  N@20=0.0012  (56s)
  New best  R@20=0.0016  N@20=0.0012
Ep   2  loss=1.3002  R@20=0.0027  N@20=0.0022  (55s)
  New best  R@20=0.0027  N@20=0.0022
Ep   3  loss=1.2912  R@20=0.0035  N@20=0.0028  (55s)
  New best  R@20=0.0035  N@20=0.0028
Ep   4  loss=1.2852  R@20=0.0038  N@20=0.0032  (55s)
  New best  R@20=0.0038  N@20=0.0032
Ep   5  loss=1.2806  R@20=0.0044  N@20=0.0036  (55s)
  New best  R@20=0.0044  N@20=0.0036
Ep   6  loss=1.2779  R@20=0.0048  N@20=0.0039  (55s)
  New best  R@20=0.0048  N@20=0.0039
Ep   7  loss=1.2749  R@20=0.0049  N@20=0.0040  (55s)
  New best  R@20=0.0049  N@20=0.0040
Ep   8  loss=1.2722  R@20=0.0050  N@20=0.0041  (55s)
  New best  R@20=0.0050  N@20=0.0041
Ep   9  loss=1.2705  R@20=0.0052  N@20=0.0042 

## Cell 8 — Results Summary Table

In [11]:
DATASET_LABELS = {
    'yelp':    'Yelp2018',
    'gowalla': 'Gowalla',
    'amazon':  'Amazon-Book',
    'ml-10m':  'ML-10M',
    'tmall':   'Tmall',
}

print(f"\n{'='*50}")
print(f"SemLightGCL (full) Results")
print(f"{'='*50}")
print(f"{'Dataset':<14} {'Recall@20':>12} {'NDCG@20':>12}")
print(f"{'-'*50}")
for ds in ['yelp', 'gowalla', 'amazon', 'ml-10m', 'tmall']:
    val = results_store.get((ds, 'full'))
    label = DATASET_LABELS[ds]
    if val:
        r, n = val
        print(f"{label:<14} {r:>12.4f} {n:>12.4f}")
    else:
        print(f"{label:<14} {'(missing)':>12}")
print(f"{'='*50}")


SemLightGCL (full) Results
Dataset           Recall@20      NDCG@20
--------------------------------------------------
Yelp2018             0.0055       0.0044
Gowalla              0.1753       0.1038
Amazon-Book          0.0798       0.0624
ML-10M            (missing)
Tmall             (missing)
